In [1]:
pip install lifetimes xgboost scikit-learn statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 14.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

def _resolve_data_root() -> Path:
    candidates = [
        Path('/kaggle/input/competitions/datathon-2026-round-1'),
        Path('/kaggle/input/datathon-2026-round-1'),
        Path.cwd() / 'data' / 'data_cleaned',
        Path.cwd().parent / 'data' / 'data_cleaned',
    ]
    for p in candidates:
        if (p / 'orders.csv').exists():
            return p
    raise FileNotFoundError(
        'Khong tim thay orders.csv. Dat data vao data/data_cleaned/ hoac mo notebook tu root repo.'
    )

DATA_ROOT = _resolve_data_root()

# =============================================================================
# LOAD DATA DUNG CHUNG
# =============================================================================
orders      = pd.read_csv(DATA_ROOT / 'orders.csv', parse_dates=['order_date'])
payments    = pd.read_csv(DATA_ROOT / 'payments.csv')
customers   = pd.read_csv(DATA_ROOT / 'customers.csv')
order_items = pd.read_csv(
    DATA_ROOT / 'order_items.csv',
    dtype={'promo_id': str, 'promo_id_2': str},
)
products    = pd.read_csv(DATA_ROOT / 'products.csv')
web_traffic = pd.read_csv(DATA_ROOT / 'web_traffic.csv', parse_dates=['date'])
sales       = pd.read_csv(DATA_ROOT / 'sales.csv')
geography   = pd.read_csv(DATA_ROOT / 'geography.csv')
promotions  = pd.read_csv(DATA_ROOT / 'promotions.csv', parse_dates=['start_date', 'end_date'])
 
# Gộp payment → giá trị đơn hàng
tx_data = orders.merge(
    payments.groupby('order_id')['payment_value'].sum().reset_index(),
    on='order_id', how='left'
)
tx_data['order_date'] = pd.to_datetime(tx_data['order_date'])
 
# orders cũng cần payment_value cho Pillar 5/8
orders = tx_data.copy()
 
# Enrich order_items: thêm category, price, net_revenue (1 lần duy nhất)
order_items = order_items.merge(
    products[['product_id', 'category', 'price']], on='product_id', how='left'
)
order_items['discount_amount'] = order_items['discount_amount'].fillna(0)
order_items['net_revenue'] = (
    order_items['unit_price'] * order_items['quantity']
) - order_items['discount_amount']
 
MAX_DATE = orders['order_date'].max()
MIN_DATE = orders['order_date'].min()
 
print('[OK] Data loaded.')
print(f'   DATA_ROOT: {DATA_ROOT}')
print(f'   Date range: {MIN_DATE.date()} -> {MAX_DATE.date()}')
print(f'   orders: {len(orders):,} | order_items: {len(order_items):,} | customers: {len(customers):,}')

✅ Data loaded.
   Date range: 2012-07-04 → 2022-12-31
   orders: 646,945 | order_items: 714,669 | customers: 121,930


# 1. Pillar 1: Tính toán LTV (Lifetimes) và Ma trận Churn

In [3]:
# =============================================================================
# PILLAR 1: MA TRẬN PHÂN BỔ NGÂN SÁCH GIỮ CHÂN
# =============================================================================
print("\n" + "="*80)
print("PILLAR 1: LTV (BG/NBD + Gamma-Gamma) & CHURN MATRIX")
print("="*80)
 
max_date = tx_data['order_date'].max()
 
# ------------------------------------------------------------------
# [P1-BUG1 FIX] Tách dữ liệu theo 2 giai đoạn rõ ràng
#   - Feature window : toàn bộ lịch sử ĐẾN TRƯỚC cutoff_date
#   - Label window   : 90 ngày SAU cutoff_date
# Điều này tránh recency (= số ngày từ last_order đến max_date)
# encode ngầm thông tin về việc KH có mua trong 90 ngày cuối không.
# ------------------------------------------------------------------
CHURN_WINDOW_DAYS = 90
cutoff_date = max_date - pd.Timedelta(days=CHURN_WINDOW_DAYS)
 
# Dữ liệu để tính RFM (chỉ dùng giao dịch TRƯỚC cutoff)
tx_feature = tx_data[tx_data['order_date'] < cutoff_date].copy()
tx_feature['order_date_only'] = tx_feature['order_date'].dt.date
 
# Dữ liệu để gán nhãn churn (giao dịch TRONG 90 ngày cuối)
buyers_in_label_window = set(
    tx_data[tx_data['order_date'] >= cutoff_date]['customer_id'].unique()
)
 
# ------------------------------------------------------------------
# BG/NBD + Gamma-Gamma: Fit trên feature window
# ------------------------------------------------------------------
rfm_all = summary_data_from_transaction_data(
    tx_feature,
    customer_id_col='customer_id',
    datetime_col='order_date_only',
    monetary_value_col='payment_value',
    observation_period_end=cutoff_date.date()
)
 
# Gamma-Gamma chỉ áp dụng cho repeat buyers (frequency > 0)
rfm_repeat = rfm_all[rfm_all['frequency'] > 0].copy()
 
# Kiểm tra assumption Gamma-Gamma: corr(frequency, monetary) phải thấp
corr_fm = rfm_repeat[['frequency', 'monetary_value']].corr().iloc[0, 1]
print(f"\n[Assumption Check] Corr(frequency, monetary_value) = {corr_fm:.3f}")
if abs(corr_fm) > 0.3:
    print("  ⚠️  Tương quan cao (>0.3) — Gamma-Gamma assumption bị vi phạm, kết quả LTV có thể lệch.")
else:
    print("  ✅ Assumption Gamma-Gamma OK.")
 
# Fit BG/NBD (penalizer để tránh overfitting khi data thưa)
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(rfm_repeat['frequency'], rfm_repeat['recency'], rfm_repeat['T'])
 
rfm_repeat['predicted_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    365, rfm_repeat['frequency'], rfm_repeat['recency'], rfm_repeat['T']
)
 
# Fit Gamma-Gamma
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(rfm_repeat['frequency'], rfm_repeat['monetary_value'])
 
rfm_repeat['predicted_LTV_12m'] = ggf.customer_lifetime_value(
    bgf,
    rfm_repeat['frequency'],
    rfm_repeat['recency'],
    rfm_repeat['T'],
    rfm_repeat['monetary_value'],
    time=12,
    discount_rate=0.01
)
 
# ------------------------------------------------------------------
# Gán nhãn churn (dựa trên label window, KHÔNG dùng recency làm feature)
# ------------------------------------------------------------------
rfm_repeat_reset = rfm_repeat.reset_index()
rfm_repeat_reset['is_churn'] = (~rfm_repeat_reset['customer_id'].isin(buyers_in_label_window)).astype(int)
 
# Features cho XGBoost: dùng frequency, T, monetary — KHÔNG dùng recency
# vì recency của feature window đã an toàn (không leak label window)
# Nhưng để hoàn toàn sạch, ta dùng T (age of customer) thay cho recency
feature_cols = ['frequency', 'T', 'monetary_value']
X = rfm_repeat_reset[feature_cols]
y = rfm_repeat_reset['is_churn']
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
 
xgb = XGBClassifier(eval_metric='logloss', n_estimators=100, max_depth=4, random_state=42)
xgb.fit(X_train, y_train)
 
# [P1-LOW FIX] In AUC và accuracy để đánh giá chất lượng model
y_pred_proba = xgb.predict_proba(X_test)[:, 1]
y_pred_label = xgb.predict(X_test)
print(f"\n[Model Quality] XGBoost Churn Model:")
print(f"  AUC-ROC  : {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_label):.4f}")
 
# Predict trên toàn bộ tập (dùng feature window)
rfm_repeat_reset['churn_probability'] = xgb.predict_proba(X)[:, 1]
 
# ------------------------------------------------------------------
# [P1-BUG2 FIX] Tính percentile trên toàn bộ KH (all_customers_ltv)
# bằng cách gán LTV = 0 cho one-time buyers (không có trong rfm_repeat)
# ------------------------------------------------------------------
all_customers = rfm_all.reset_index()[['customer_id']].copy()
all_customers = all_customers.merge(
    rfm_repeat_reset[['customer_id', 'predicted_LTV_12m']],
    on='customer_id', how='left'
)
all_customers['predicted_LTV_12m'] = all_customers['predicted_LTV_12m'].fillna(0)
 
# Các ngưỡng phân vị tính trên TOÀN BỘ KH (bao gồm cả one-time buyers)
p99_global = all_customers['predicted_LTV_12m'].quantile(0.99)
p90_global = all_customers['predicted_LTV_12m'].quantile(0.90)
p75_global = all_customers['predicted_LTV_12m'].quantile(0.75)
p50_global = all_customers['predicted_LTV_12m'].quantile(0.50)  # thêm cho tier Bronze
 
churn_threshold = 0.6
 
# ------------------------------------------------------------------
# [P1-BUG3 FIX] Thêm tier "Bronze" để không bỏ sót mid-LTV high-churn
# ------------------------------------------------------------------
def assign_tier(row):
    ltv   = row['predicted_LTV_12m']
    churn = row['churn_probability']
 
    if churn >= churn_threshold:
        # High Churn Risk → ưu tiên can thiệp theo giá trị LTV
        if ltv >= p99_global:
            return "Priority 1A - Diamond (Top 1%): Telesales 1-1 / Quà VIP"
        elif ltv >= p90_global:
            return "Priority 1B - Gold (Top 2-10%): SMS/Zalo + Voucher định danh"
        elif ltv >= p75_global:
            return "Priority 1C - Silver (Top 11-25%): App Push/Email + Discount nhẹ"
        elif ltv >= p50_global:
            # [NEW] Mid-LTV không bị bỏ qua hoàn toàn
            return "Priority 1D - Bronze (Top 26-50%): Email Trigger + Loyalty Point"
        else:
            # Chỉ bỏ qua nhóm bottom 50% LTV + high churn (thực sự low value)
            return "Skip: Để Churn tự nhiên"
    else:
        # Low Churn Risk → nuôi dưỡng
        if ltv >= p75_global:
            return "Priority 2: Nuôi dưỡng Email Automation"
        else:
            return "Monitor: Theo dõi thêm"
 
churn_df = rfm_repeat_reset.copy()
churn_df['Prescriptive_Tier'] = churn_df.apply(assign_tier, axis=1)
 
# ------------------------------------------------------------------
# Bảng tổng hợp
# ------------------------------------------------------------------
summary_tier = churn_df.groupby('Prescriptive_Tier').agg(
    So_luong_Khach       = ('customer_id', 'count'),
    Tong_LTV_Ky_Vong_12T = ('predicted_LTV_12m', 'sum'),
    Churn_Risk_TB        = ('churn_probability', 'mean')
).reset_index()
 
total_kh  = summary_tier['So_luong_Khach'].sum()
total_ltv = summary_tier['Tong_LTV_Ky_Vong_12T'].sum()
summary_tier['% KH']  = (summary_tier['So_luong_Khach'] / total_kh * 100).round(2)
summary_tier['% LTV'] = (summary_tier['Tong_LTV_Ky_Vong_12T'] / total_ltv * 100).round(2)
summary_tier = summary_tier.sort_values('% LTV', ascending=False)
summary_tier['Tong_LTV_Ky_Vong_12T'] = summary_tier['Tong_LTV_Ky_Vong_12T'].apply(lambda x: f"{x:,.0f}")
summary_tier['Churn_Risk_TB'] = summary_tier['Churn_Risk_TB'].round(4)
 
print("\n" + "="*80)
print("BẢNG TỔNG HỢP CHIẾN LƯỢC RETENTION")
print("="*80)
print(summary_tier[['Prescriptive_Tier', 'So_luong_Khach', '% KH',
                     'Tong_LTV_Ky_Vong_12T', '% LTV', 'Churn_Risk_TB']].to_string(index=False))


PILLAR 1: LTV (BG/NBD + Gamma-Gamma) & CHURN MATRIX

[Assumption Check] Corr(frequency, monetary_value) = -0.006
  ✅ Assumption Gamma-Gamma OK.

[Model Quality] XGBoost Churn Model:
  AUC-ROC  : 0.7093
  Accuracy : 0.9231

BẢNG TỔNG HỢP CHIẾN LƯỢC RETENTION
                                               Prescriptive_Tier  So_luong_Khach  % KH Tong_LTV_Ky_Vong_12T  % LTV  Churn_Risk_TB
    Priority 1B - Gold (Top 2-10%): SMS/Zalo + Voucher định danh            7983 11.82          409,635,964  32.04         0.8246
Priority 1C - Silver (Top 11-25%): App Push/Email + Discount nhẹ           13502 19.99          377,575,627  29.53         0.8983
Priority 1D - Bronze (Top 26-50%): Email Trigger + Loyalty Point           22512 33.33          309,934,363  24.24         0.9427
                                         Skip: Để Churn tự nhiên           22511 33.32           82,466,433   6.45         0.9678
         Priority 1A - Diamond (Top 1%): Telesales 1-1 / Quà VIP             611  0.90     

# Pillar 2: Độ đàn hồi giá (Price Elasticity) để tìm Breakeven Discount

In [4]:
# =============================================================================
# PILLAR 2: TỐI ƯU LỢI NHUẬN QUA RÀO CHẮN KHUYẾN MÃI (BREAKEVEN DISCOUNT CAP)
# =============================================================================
print("\n\n" + "="*80)
print("PILLAR 2: PRICE ELASTICITY & HARD CAP POLICY")
print("="*80)
 
# order_items đã có 'category' và 'price' từ data loader — không merge lại
df_items = order_items.copy()
 
# Tính % discount và giá thực tế
df_items['price'] = df_items['price'].replace(0, np.nan)
df_items['discount_pct'] = (df_items['discount_amount'] / df_items['price']).fillna(0)
df_items['price_actual'] = df_items['price'] * (1 - df_items['discount_pct'])
 
# Lọc dữ liệu hợp lệ
df_items = df_items[
    (df_items['discount_pct'] >= 0) &
    (df_items['discount_pct'] < 1.0) &
    (df_items['price_actual'] > 0) &
    (df_items['quantity'] > 0)
].copy()
 
# Hàm tính elasticity cho 1 category
def calc_elasticity(df_cat, category_name):
    """
    [P2-BUG4 FIX] Log-log regression chuẩn:
      ln(quantity) ~ ln(price_actual)
      Hệ số của ln(price_actual) chính là price elasticity E.
    """
    grp = df_cat.groupby('discount_pct').agg(
        quantity     = ('quantity', 'sum'),
        price_actual = ('price_actual', 'mean')  # giá trung bình tại mỗi mức discount
    ).reset_index()
 
    # Lọc noise
    grp = grp[grp['quantity'] > 0].replace([np.inf, -np.inf], np.nan).dropna()
 
    # [P2-BUG5] Cảnh báo nếu quá ít data points
    n_points = len(grp)
    if n_points < 10:
        print(f"  ⚠️  [{category_name}] Chỉ có {n_points} data points — kết quả elasticity kém tin cậy.")
    else:
        print(f"  ✅ [{category_name}] {n_points} data points — đủ để hồi quy.")
 
    grp['ln_quantity']     = np.log(grp['quantity'])
    grp['ln_price_actual'] = np.log(grp['price_actual'])
 
    X = sm.add_constant(grp['ln_price_actual'])
    y = grp['ln_quantity']
    model = sm.OLS(y, X).fit()
 
    elasticity = model.params['ln_price_actual']
    p_value    = model.pvalues['ln_price_actual']
    r_squared  = model.rsquared
    return elasticity, p_value, r_squared
 
for cat in ['Outdoor', 'Streetwear']:
    df_cat = df_items[df_items['category'] == cat]
    if df_cat.empty:
        print(f"\n  [SKIP] Không tìm thấy category '{cat}' trong data.")
        continue
 
    e, p, r2 = calc_elasticity(df_cat, cat)
    print(f"\n  Category: {cat}")
    print(f"  Elasticity = {e:.2f}  |  p-value = {p:.3f}  |  R² = {r2:.3f}")
 
    if p > 0.05:
        print(f"  ⚠️  p-value > 0.05 — hệ số không có ý nghĩa thống kê, cẩn thận khi ra policy.")
 
    if e > -1:
        print(f"  → Inelastic: giảm giá KHÔNG kéo đủ volume bù AOV.")
        cap = 5 if cat == 'Outdoor' else 10
        print(f"  → PRESCRIPTIVE: Hard Cap discount tối đa {cap}% cho {cat}.")
    else:
        print(f"  → Elastic: giảm giá kéo được volume.")
        print(f"  → PRESCRIPTIVE: Có thể dùng promotion có kiểm soát, nhưng theo dõi margin.")




PILLAR 2: PRICE ELASTICITY & HARD CAP POLICY
  ✅ [Outdoor] 90032 data points — đủ để hồi quy.

  Category: Outdoor
  Elasticity = -0.30  |  p-value = 0.000  |  R² = 0.276
  → Inelastic: giảm giá KHÔNG kéo đủ volume bù AOV.
  → PRESCRIPTIVE: Hard Cap discount tối đa 5% cho Outdoor.
  ✅ [Streetwear] 115802 data points — đủ để hồi quy.

  Category: Streetwear
  Elasticity = -0.39  |  p-value = 0.000  |  R² = 0.223
  → Inelastic: giảm giá KHÔNG kéo đủ volume bù AOV.
  → PRESCRIPTIVE: Hard Cap discount tối đa 10% cho Streetwear.


# Pillar 3: Tái phân bổ ngân sách (MMM Attribution vs. Retention)

In [5]:
# =============================================================================
# PILLAR 3: TÁI CẤU TRÚC MEDIA MIX (MMM vs. RETENTION)
# =============================================================================
print("\n\n" + "="*80)
print("PILLAR 3: MARKETING MIX MODELING (MMM)")
print("="*80)
 
web_traffic['date'] = pd.to_datetime(web_traffic['date'])
sales['Date']       = pd.to_datetime(sales['Date'])
 
traffic_pivot = web_traffic.pivot_table(
    index='date', columns='traffic_source', values='sessions', aggfunc='sum'
).reset_index().fillna(0)
 
# [P3-BUG7 FIX] Adstock transformation (geometric decay)
# Hiệu quả quảng cáo kéo dài nhiều ngày, không chỉ trong ngày hôm đó.
# Adstock(t) = sessions(t) + decay_rate * Adstock(t-1)
ADSTOCK_DECAY = 0.5  # Mỗi ngày giữ lại 50% hiệu quả hôm trước (có thể tune)
 
channel_cols = [c for c in traffic_pivot.columns if c != 'date']
 
def apply_adstock(series, decay=ADSTOCK_DECAY):
    adstocked = np.zeros(len(series))
    for i, val in enumerate(series):
        adstocked[i] = val + (decay * adstocked[i-1] if i > 0 else 0)
    return adstocked
 
traffic_adstocked = traffic_pivot[['date']].copy()
for col in channel_cols:
    traffic_adstocked[col + '_adstock'] = apply_adstock(traffic_pivot[col].values)
 
mmm_data = sales.merge(traffic_adstocked, left_on='Date', right_on='date', how='inner')
 
adstock_cols = [c for c in mmm_data.columns if c.endswith('_adstock')]
X_mmm = sm.add_constant(mmm_data[adstock_cols])
y_mmm = mmm_data['Revenue']
 
mmm_model = sm.OLS(y_mmm, X_mmm).fit()
 
print("\nKẾT QUẢ PREDICTIVE: HỆ SỐ MARGINAL REVENUE (MMM với Adstock)")
print("-"*70)
for col in adstock_cols:
    original_channel = col.replace('_adstock', '')
    coef  = mmm_model.params[col]
    p_val = mmm_model.pvalues[col]
    sig   = "✅" if p_val < 0.05 else "⚠️ (không có ý nghĩa thống kê)"
    print(f"  {original_channel:20s}: ~{coef:,.0f} VNĐ / session  (p={p_val:.3f}) {sig}")
 
print(f"\n  R² tổng thể của MMM model: {mmm_model.rsquared:.4f}")
 
# [P3-BUG6 FIX] Tự động tìm cột paid thay vì hardcode 'Paid'
paid_cols = [c for c in adstock_cols if 'paid' in c.lower()]
print(f"\n  [DEBUG] Cột kênh Paid được detect: {paid_cols}")
 
print("\n" + "="*70)
print("PRESCRIPTIVE ACTION: ĐIỀU HƯỚNG NGÂN SÁCH")
print("="*70)
 
if paid_cols:
    paid_col          = paid_cols[0]
    marginal_rev_paid = mmm_model.params[paid_col]  # doanh thu tăng thêm / 1 session
    p_paid            = mmm_model.pvalues[paid_col]
 
    # CPC thực tế lấy từ marketing report (đơn vị: VNĐ / session)
    CPC_PAID = 5000  # ← Cập nhật con số thực từ Google Ads / Facebook Ads report
    roi_paid = (marginal_rev_paid - CPC_PAID) / CPC_PAID
 
    print(f"  Marginal Revenue của {paid_col.replace('_adstock','')}: {marginal_rev_paid:,.0f} VNĐ/session")
    print(f"  CPC giả định                                         : {CPC_PAID:,} VNĐ/session")
    print(f"  → Marginal ROI kênh Paid: {roi_paid*100:.1f}%")
 
    if p_paid > 0.05:
        print("  ⚠️  Hệ số kênh Paid không có ý nghĩa thống kê — kết luận ROI cần thận trọng.")
 
    if roi_paid < 0.5:
        print("  → KẾT LUẬN: Kênh Paid có dấu hiệu Diminishing Returns.")
        print("  → HÀNH ĐỘNG: Freeze ngân sách Paid. Chuyển phần dư sang Retention Priority 1 (Pillar 1).")
    else:
        print("  → KẾT LUẬN: Kênh Paid vẫn sinh lời tốt ở điểm biên.")
        print("  → HÀNH ĐỘNG: Duy trì hoặc scale có kiểm soát.")
else:
    print("  ⚠️  Không tìm thấy cột kênh Paid trong data. Kiểm tra lại tên traffic_source.")



PILLAR 3: MARKETING MIX MODELING (MMM)

KẾT QUẢ PREDICTIVE: HỆ SỐ MARGINAL REVENUE (MMM với Adstock)
----------------------------------------------------------------------
  direct              : ~46 VNĐ / session  (p=0.000) ✅
  email_campaign      : ~46 VNĐ / session  (p=0.000) ✅
  organic_search      : ~42 VNĐ / session  (p=0.000) ✅
  paid_search         : ~50 VNĐ / session  (p=0.000) ✅
  referral            : ~45 VNĐ / session  (p=0.000) ✅
  social_media        : ~55 VNĐ / session  (p=0.000) ✅

  R² tổng thể của MMM model: 0.1094

  [DEBUG] Cột kênh Paid được detect: ['paid_search_adstock']

PRESCRIPTIVE ACTION: ĐIỀU HƯỚNG NGÂN SÁCH
  Marginal Revenue của paid_search: 50 VNĐ/session
  CPC giả định                                         : 5,000 VNĐ/session
  → Marginal ROI kênh Paid: -99.0%
  → KẾT LUẬN: Kênh Paid có dấu hiệu Diminishing Returns.
  → HÀNH ĐỘNG: Freeze ngân sách Paid. Chuyển phần dư sang Retention Priority 1 (Pillar 1).


# Pillar 4: Kịch bản đa dạng hóa (Scenario Forecasting)

In [6]:
# =============================================================================
# PILLAR 4: ĐỊNH LƯỢNG RỦI RO TẬP TRUNG & KỊCH BẢN DOANH THU
# =============================================================================
print("\n\n" + "="*80)
print("PILLAR 4: SCENARIO FORECASTING & DIVERSIFICATION KPI")
print("="*80)
 
# order_items đã có 'category', 'net_revenue' từ data loader
# Chỉ cần join thêm order_date từ orders
df_merged = order_items.merge(orders[['order_id', 'order_date']], on='order_id', how='left')
df_merged['order_date'] = pd.to_datetime(df_merged['order_date'])
 
# Baseline: 6 tháng gần nhất
max_date_p4   = df_merged['order_date'].max()
last_6_months = df_merged[df_merged['order_date'] >= (max_date_p4 - pd.DateOffset(months=6))]
 
# [P4-BUG8 FIX] Tính doanh thu TRUNG BÌNH THÁNG (baseline 1 tháng)
monthly_rev       = last_6_months.groupby('category')['net_revenue'].sum() / 6
total_monthly_rev = monthly_rev.sum()
 
# Auto-detect category chính và phụ
cat_main = 'Streetwear' if 'Streetwear' in monthly_rev.index else monthly_rev.idxmax()
cat_sub  = 'Outdoor'    if 'Outdoor'    in monthly_rev.index else monthly_rev.drop(cat_main).idxmax()
 
rev_main_base = monthly_rev.get(cat_main, 0)
rev_sub_base  = monthly_rev.get(cat_sub, 0)
 
print(f"\nBaseline Doanh thu trung bình (1 tháng): {total_monthly_rev:,.0f} VNĐ")
print(f"  - {cat_main}: {rev_main_base:,.0f} VNĐ ({rev_main_base/total_monthly_rev*100:.1f}%)")
print(f"  - {cat_sub} : {rev_sub_base:,.0f} VNĐ ({rev_sub_base/total_monthly_rev*100:.1f}%)")
 
# Kịch bản xấu
for drop_rate in [0.10, 0.20]:
    revenue_gap = rev_main_base * drop_rate
 
    # [P4-BUG8 FIX] KPI bù đắp tính NHẤT QUÁN theo từng tháng
    # Gap = doanh thu thiếu hụt mỗi tháng
    # Target tăng trưởng của cat_sub cũng tính theo tháng
    required_growth_pct     = revenue_gap / rev_sub_base          # % tăng / tháng
    required_growth_monthly = rev_sub_base * required_growth_pct  # VNĐ tăng thêm / tháng
 
    # Quy đổi sang 2 quý (6 tháng) — nhất quán đơn vị
    # Giả định tăng trưởng tuyến tính trong 6 tháng
    monthly_ramp = required_growth_monthly / 6  # Mỗi tháng cần tăng thêm bao nhiêu
    total_6m_gap = revenue_gap * 6              # Tổng gap cần bù trong 6 tháng
 
    print(f"\n{'='*60}")
    print(f"KỊCH BẢN: {cat_main} sụt {drop_rate*100:.0f}% / tháng")
    print(f"{'='*60}")
    print(f"  Gap doanh thu mỗi tháng : {revenue_gap:,.0f} VNĐ")
    print(f"  Gap tích lũy trong 6 tháng: {total_6m_gap:,.0f} VNĐ")
    print(f"\n  PRESCRIPTIVE — KPI cho {cat_sub}:")
    print(f"  → Cần tăng trưởng: +{required_growth_pct*100:.1f}% / tháng so với baseline")
    print(f"  → Tương đương: +{required_growth_monthly:,.0f} VNĐ / tháng")
    print(f"  → Để đạt trong 2 quý (6 tháng): tăng dần ~{monthly_ramp:,.0f} VNĐ / tháng")
    print(f"  → Chiến lược: Bundling Streetwear + {cat_sub} (Mồi nhử) đã validate ở Q3.")



PILLAR 4: SCENARIO FORECASTING & DIVERSIFICATION KPI

Baseline Doanh thu trung bình (1 tháng): 74,892,499 VNĐ
  - Streetwear: 62,322,110 VNĐ (83.2%)
  - Outdoor : 7,176,720 VNĐ (9.6%)

KỊCH BẢN: Streetwear sụt 10% / tháng
  Gap doanh thu mỗi tháng : 6,232,211 VNĐ
  Gap tích lũy trong 6 tháng: 37,393,266 VNĐ

  PRESCRIPTIVE — KPI cho Outdoor:
  → Cần tăng trưởng: +86.8% / tháng so với baseline
  → Tương đương: +6,232,211 VNĐ / tháng
  → Để đạt trong 2 quý (6 tháng): tăng dần ~1,038,702 VNĐ / tháng
  → Chiến lược: Bundling Streetwear + Outdoor (Mồi nhử) đã validate ở Q3.

KỊCH BẢN: Streetwear sụt 20% / tháng
  Gap doanh thu mỗi tháng : 12,464,422 VNĐ
  Gap tích lũy trong 6 tháng: 74,786,531 VNĐ

  PRESCRIPTIVE — KPI cho Outdoor:
  → Cần tăng trưởng: +173.7% / tháng so với baseline
  → Tương đương: +12,464,422 VNĐ / tháng
  → Để đạt trong 2 quý (6 tháng): tăng dần ~2,077,404 VNĐ / tháng
  → Chiến lược: Bundling Streetwear + Outdoor (Mồi nhử) đã validate ở Q3.


#  PILLAR 5: CR RECOVERY — DỰ BÁO XÁC SUẤT CHUYỂN ĐỔI THEO PHÂN KHÚC

In [7]:
# =============================================================================
# PILLAR 5: CR RECOVERY — DỰ BÁO XÁC SUẤT CHUYỂN ĐỔI THEO PHÂN KHÚC
# =============================================================================
# Insight Q2: CR gãy từ 2019 (Z-test p=0.000000). AOV đang bù tạm thời.
# Mục tiêu: Tìm phân khúc nào có "conversion gap" lớn nhất →
#           ưu tiên action cụ thể thay vì tối ưu mù.
#
# Logic xây dựng:
#   - Mỗi session web = 1 observation
#   - Label: session đó có dẫn đến 1 đơn hàng không? (1/0)
#   - Feature: traffic_source, device_type (nếu có), day_of_week,
#              is_weekend, customer_type (new/returning)
#   - Output: xác suất chuyển đổi theo từng phân khúc → tìm nhóm
#             có actual CR thấp hơn predicted CR nhiều nhất
# =============================================================================
print("\n" + "="*80)
print("PILLAR 5: CONVERSION RATE RECOVERY")
print("="*80)
 
# ------------------------------------------------------------------
# Bước 1: Tạo session-level dataset
# Mỗi ngày + kênh = N sessions; trong đó bao nhiêu session → order?
# Dùng số đơn hàng thực tế làm proxy cho "converted sessions"
# ------------------------------------------------------------------
web_traffic['year']        = web_traffic['date'].dt.year
web_traffic['month']       = web_traffic['date'].dt.month
web_traffic['day_of_week'] = web_traffic['date'].dt.dayofweek   # 0=Mon, 6=Sun
web_traffic['is_weekend']  = (web_traffic['day_of_week'] >= 5).astype(int)
web_traffic['is_post2019'] = (web_traffic['year'] >= 2019).astype(int)
 
# Đơn hàng mỗi ngày
orders_per_day = orders.groupby(
    orders['order_date'].dt.date
)['order_id'].count().reset_index()
orders_per_day.columns = ['date', 'orders_count']
orders_per_day['date'] = pd.to_datetime(orders_per_day['date'])
 
# Gộp sessions và orders theo ngày + kênh
session_order = web_traffic.merge(
    orders_per_day, on='date', how='left'
).fillna({'orders_count': 0})
 
# Tính conversion rate thực tế mỗi ngày×kênh
# (Chia đều orders cho các kênh theo tỷ lệ sessions — best proxy khi
#  không có session_id level tracking)
total_sessions_per_day = session_order.groupby('date')['sessions'].transform('sum')
session_order['channel_share'] = session_order['sessions'] / total_sessions_per_day.replace(0, np.nan)
session_order['estimated_orders'] = session_order['orders_count'] * session_order['channel_share']
session_order['cr_actual'] = (
    session_order['estimated_orders'] / session_order['sessions'].replace(0, np.nan)
).fillna(0).clip(0, 1)
 
# ------------------------------------------------------------------
# Bước 2: Predictive — hồi quy OLS để tách effect của từng yếu tố
# lên CR (log-odds để giữ CR trong [0,1])
# ------------------------------------------------------------------
session_order_clean = session_order[session_order['sessions'] > 0].copy()
session_order_clean['cr_clipped'] = session_order_clean['cr_actual'].clip(1e-4, 1 - 1e-4)
session_order_clean['logit_cr']   = np.log(
    session_order_clean['cr_clipped'] / (1 - session_order_clean['cr_clipped'])
)
 
# Dummy encode traffic_source
source_dummies = pd.get_dummies(
    session_order_clean['traffic_source'], prefix='src', drop_first=True
)
session_order_clean = pd.concat([session_order_clean, source_dummies], axis=1)
 
feature_cols_cr = ['is_weekend', 'is_post2019', 'sessions'] + list(source_dummies.columns)
X_cr = sm.add_constant(session_order_clean[feature_cols_cr].astype(float))
y_cr = session_order_clean['logit_cr']
 
ols_cr = sm.OLS(y_cr, X_cr).fit()
 
print("\n[Predictive] Hồi quy các yếu tố tác động đến Conversion Rate (logit scale):")
print(f"  R² = {ols_cr.rsquared:.4f}")
print()
for var in feature_cols_cr:
    coef  = ols_cr.params[var]
    pval  = ols_cr.pvalues[var]
    sig   = "✅" if pval < 0.05 else "⚠️ (n.s.)"
    # Chuyển logit coefficient → delta CR tại CR baseline ~3% (mid-range)
    direction = "↑" if coef > 0 else "↓"
    print(f"  {var:30s}: coef={coef:+.3f}  p={pval:.3f}  {sig}  {direction}")
 
# ------------------------------------------------------------------
# Bước 3: Tính CR predicted vs actual theo phân khúc kênh × giai đoạn
# "Conversion gap" = actual CR thấp hơn baseline bao nhiêu?
# ------------------------------------------------------------------
segment_cr = session_order_clean.groupby(
    ['traffic_source', 'is_post2019']
).agg(
    sessions_total     = ('sessions', 'sum'),
    orders_total       = ('estimated_orders', 'sum'),
    cr_mean            = ('cr_actual', 'mean'),
    cr_std             = ('cr_actual', 'std')
).reset_index()
 
# Baseline CR = trung bình giai đoạn pre-2019 (trước khi gãy)
baseline_cr_by_source = segment_cr[segment_cr['is_post2019'] == 0].set_index('traffic_source')['cr_mean']
segment_cr['cr_baseline']    = segment_cr['traffic_source'].map(baseline_cr_by_source)
segment_cr['conversion_gap'] = segment_cr['cr_baseline'] - segment_cr['cr_mean']
segment_cr['gap_pct']        = (segment_cr['conversion_gap'] / segment_cr['cr_baseline'] * 100).round(1)
segment_cr['period']         = segment_cr['is_post2019'].map({0: 'Pre-2019', 1: 'Post-2019'})
 
post2019_gaps = segment_cr[segment_cr['is_post2019'] == 1].sort_values('conversion_gap', ascending=False)
 
print("\n[Predictive] Conversion Gap theo kênh (Post-2019 so với baseline Pre-2019):")
print("-"*75)
print(f"  {'Kênh':<20} {'CR Pre':<12} {'CR Post':<12} {'Gap (pp)':<12} {'Gap (%)':<10}")
print("-"*75)
for _, row in post2019_gaps.iterrows():
    pre  = row['cr_baseline']
    post = row['cr_mean']
    if pd.isna(pre):
        continue
    gap_pp = (pre - post) * 100
    gap_pc = row['gap_pct']
    print(f"  {row['traffic_source']:<20} {pre*100:.2f}%       {post*100:.2f}%       {gap_pp:+.2f}pp     {gap_pc:+.1f}%")
 
# ------------------------------------------------------------------
# Bước 4: Tính AOV threshold — CR cần phải giữ ở đâu để không mất doanh thu
# dù AOV tiếp tục tăng
# Công thức: Revenue = CR × Sessions × AOV
# Giữ Revenue = const → CR_floor = Revenue_target / (Sessions × AOV_current)
# ------------------------------------------------------------------
aov_pre  = orders[orders['order_date'].dt.year < 2019]['payment_value'].mean()
aov_post = orders[orders['order_date'].dt.year >= 2019]['payment_value'].mean()
 
sessions_post = session_order_clean[session_order_clean['is_post2019'] == 1]['sessions'].sum()
sessions_pre  = session_order_clean[session_order_clean['is_post2019'] == 0]['sessions'].sum()
cr_pre_mean   = session_order_clean[session_order_clean['is_post2019'] == 0]['cr_actual'].mean()
cr_post_mean  = session_order_clean[session_order_clean['is_post2019'] == 1]['cr_actual'].mean()
 
revenue_pre_proxy  = sessions_pre  * cr_pre_mean  * aov_pre
revenue_post_proxy = sessions_post * cr_post_mean * aov_post
 
# CR floor: CR tối thiểu để giữ doanh thu = mức pre-2019 (khi AOV tiếp tục tăng)
cr_floor = (sessions_pre * cr_pre_mean * aov_pre) / (sessions_post * aov_post)
cr_floor = min(cr_floor, 1.0)
 
print(f"\n[Predictive] AOV trước/sau 2019:")
print(f"  AOV Pre-2019  : {aov_pre:,.0f} VNĐ")
print(f"  AOV Post-2019 : {aov_post:,.0f} VNĐ  (tăng {(aov_post/aov_pre-1)*100:.1f}%)")
print(f"\n  CR thực tế Post-2019 : {cr_post_mean*100:.3f}%")
print(f"  CR sàn để giữ doanh thu (AOV-adjusted) : {cr_floor*100:.3f}%")
if cr_post_mean >= cr_floor:
    print("  ✅ CR hiện tại đủ để bù đắp nhờ AOV tăng.")
else:
    shortfall = (cr_floor - cr_post_mean) * 100
    print(f"  ⚠️  CR thiếu hụt {shortfall:.3f}pp so với ngưỡng an toàn.")
 
# ------------------------------------------------------------------
# Prescriptive Output
# ------------------------------------------------------------------
print("\n" + "="*70)
print("PRESCRIPTIVE: ĐIỀU HƯỚNG TỐI ƯU CR")
print("="*70)
top_gap_channel = post2019_gaps.dropna(subset=['cr_baseline']).iloc[0]
print(f"""
  1. Kênh có conversion gap lớn nhất: [{top_gap_channel['traffic_source']}]
     → Đây là điểm can thiệp ưu tiên, không phải tăng traffic mà tối ưu
       landing page / checkout flow cho riêng kênh này.
 
  2. CR sàn an toàn = {cr_floor*100:.3f}%
     → Đặt làm KPI monitoring hàng tháng. Nếu CR vượt dưới ngưỡng này
       VÀ AOV không tăng tương ứng, doanh thu sẽ thực sự bị suy giảm.
 
  3. Không đổ thêm ngân sách acquisition vào kênh có gap cao nhất.
     Mỗi session mới từ kênh đó đang "rò rỉ" nhiều hơn kênh khác.
     → Ưu tiên: A/B test onsite UX cho kênh gap cao trước.
""")


PILLAR 5: CONVERSION RATE RECOVERY

[Predictive] Hồi quy các yếu tố tác động đến Conversion Rate (logit scale):
  R² = 0.5671

  is_weekend                    : coef=-0.074  p=0.000  ✅  ↓
  is_post2019                   : coef=-0.976  p=0.000  ✅  ↓
  sessions                      : coef=-0.000  p=0.000  ✅  ↓
  src_email_campaign            : coef=-0.002  p=0.946  ⚠️ (n.s.)  ↓
  src_organic_search            : coef=-0.001  p=0.965  ⚠️ (n.s.)  ↓
  src_paid_search               : coef=+0.007  p=0.826  ⚠️ (n.s.)  ↑
  src_referral                  : coef=-0.001  p=0.987  ⚠️ (n.s.)  ↓
  src_social_media              : coef=+0.005  p=0.874  ⚠️ (n.s.)  ↑

[Predictive] Conversion Gap theo kênh (Post-2019 so với baseline Pre-2019):
---------------------------------------------------------------------------
  Kênh                 CR Pre       CR Post      Gap (pp)     Gap (%)   
---------------------------------------------------------------------------
  organic_search       1.00%       0.35%  

# PILLAR 6: PROMO → LOYAL — PIPELINE CHUYỂN ĐỔI KH PROMO THÀNH LOYAL

In [8]:
# =============================================================================
# PILLAR 8: PROMO → LOYAL — PIPELINE CHUYỂN ĐỔI KH PROMO THÀNH LOYAL
# =============================================================================
# Insight Q10: Khách Loyal có retention cao hơn Promo (Z-test p=0.000003).
# Mục tiêu: Trong số KH Promo hiện tại, ai có khả năng cao trở thành Loyal?
#           → Trigger nurture không dùng discount cho nhóm đó.
#
# Định nghĩa:
#   Promo buyer  : Đã dùng promo_id trong ít nhất 1 đơn
#   Loyal buyer  : Đã mua ≥ 3 lần VÀ ít nhất 1 lần không dùng promo
#   Label        : KH Promo hiện tại → trong 90 ngày tiếp theo có trở thành
#                  Loyal không? (mua lại mà không dùng promo)
# =============================================================================
print("\n\n" + "="*80)
print("PILLAR 6: PROMO → LOYAL CONVERSION PIPELINE")
print("="*80)
 
# ------------------------------------------------------------------
# Bước 1: Gán nhãn Promo / Loyal tại cutoff
# ------------------------------------------------------------------
CUTOFF_P8     = MAX_DATE - pd.Timedelta(days=90)
LABEL_END_P8  = MAX_DATE
 
# Lịch sử TRƯỚC cutoff → để tính feature
oi_pre  = order_items.merge(orders[['order_id', 'customer_id', 'order_date']],
                             on='order_id', how='left')
oi_pre  = oi_pre[oi_pre['order_date'] < CUTOFF_P8].copy()
 
# Lịch sử TRONG label window → để gán nhãn
oi_label = order_items.merge(orders[['order_id', 'customer_id', 'order_date']],
                              on='order_id', how='left')
oi_label = oi_label[
    (oi_label['order_date'] >= CUTOFF_P8) &
    (oi_label['order_date'] <= LABEL_END_P8)
].copy()
 
# Có dùng promo trong feature window không?
oi_pre['has_promo'] = (oi_pre['promo_id'].notna() | oi_pre['promo_id_2'].notna()).astype(int)
promo_history = oi_pre.groupby('customer_id').agg(
    total_orders       = ('order_id', 'nunique'),
    promo_orders       = ('has_promo', 'sum'),
    non_promo_orders   = ('has_promo', lambda x: (x == 0).sum()),
    avg_discount_rate  = ('discount_amount', lambda x:
                          (x.sum() / (oi_pre.loc[x.index, 'unit_price'] *
                                      oi_pre.loc[x.index, 'quantity']).sum())
                          if (oi_pre.loc[x.index, 'unit_price'] *
                              oi_pre.loc[x.index, 'quantity']).sum() > 0 else 0),
    categories_bought  = ('category', 'nunique'),
    net_revenue_total  = ('net_revenue', 'sum'),
    last_order_date    = ('order_date', 'max'),
).reset_index()
 
promo_history['promo_rate']    = promo_history['promo_orders'] / promo_history['total_orders'].clip(1)
promo_history['days_since_last'] = (CUTOFF_P8 - pd.to_datetime(promo_history['last_order_date'])).dt.days
 
# Định nghĩa "Promo buyer" ở thời điểm cutoff
promo_history['is_promo_buyer'] = (
    (promo_history['promo_orders'] > 0) &
    (promo_history['total_orders'] >= 1)
).astype(int)
 
# Subset: chỉ xét KH Promo
promo_customers = promo_history[promo_history['is_promo_buyer'] == 1].copy()
 
# ------------------------------------------------------------------
# Bước 2: Gán nhãn — KH Promo có "thoát" khỏi promo trong 90 ngày tiếp không?
# Label = 1 nếu mua lại mà không dùng promo trong label window
# ------------------------------------------------------------------
if not oi_label.empty:
    oi_label['has_promo_label'] = (
        oi_label['promo_id'].notna() | oi_label['promo_id_2'].notna()
    ).astype(int)
 
    # Khách mua trong label window mà có ít nhất 1 lần KHÔNG dùng promo
    loyal_converted = oi_label.groupby('customer_id').agg(
        bought_without_promo = ('has_promo_label', lambda x: (x == 0).any())
    ).reset_index()
    loyal_converted['converted_to_loyal'] = loyal_converted['bought_without_promo'].astype(int)
 
    promo_customers = promo_customers.merge(
        loyal_converted[['customer_id', 'converted_to_loyal']],
        on='customer_id', how='left'
    )
    promo_customers['converted_to_loyal'] = promo_customers['converted_to_loyal'].fillna(0).astype(int)
else:
    # Không có data trong label window — tạo label giả để demo flow
    promo_customers['converted_to_loyal'] = 0
    print("  ⚠️  Không có giao dịch trong label window. Label = 0 cho tất cả KH.")
 
conversion_rate_actual = promo_customers['converted_to_loyal'].mean()
print(f"\n  Tỷ lệ Promo → Loyal thực tế trong 90 ngày: {conversion_rate_actual*100:.2f}%")
print(f"  Số KH Promo phân tích: {len(promo_customers):,}")
 
# ------------------------------------------------------------------
# Bước 3: Predictive — XGBoost dự báo propensity chuyển đổi
# ------------------------------------------------------------------
feature_cols_p8 = [
    'total_orders', 'promo_rate', 'non_promo_orders',
    'categories_bought', 'net_revenue_total', 'days_since_last',
    'avg_discount_rate'
]
 
p8_df = promo_customers[feature_cols_p8 + ['converted_to_loyal']].dropna()
X_p8  = p8_df[feature_cols_p8]
y_p8  = p8_df['converted_to_loyal']
 
# Kiểm tra class balance
pos_rate = y_p8.mean()
if pos_rate < 0.05 or pos_rate > 0.95:
    print(f"  ⚠️  Label imbalance: {pos_rate*100:.1f}% positive. Dùng scale_pos_weight.")
    spw = (1 - pos_rate) / pos_rate
else:
    spw = 1.0
 
if len(p8_df) < 50:
    print("  ⚠️  Quá ít data để train model đáng tin cậy. Kết quả chỉ mang tính định hướng.")
 
X_tr8, X_te8, y_tr8, y_te8 = train_test_split(
    X_p8, y_p8, test_size=0.2, random_state=42,
    stratify=y_p8 if y_p8.sum() > 5 else None
)
 
xgb_p8 = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw, eval_metric='auc', random_state=42
)
xgb_p8.fit(X_tr8, y_tr8)
 
y_prob_p8 = xgb_p8.predict_proba(X_te8)[:, 1]
auc_p8    = roc_auc_score(y_te8, y_prob_p8) if y_te8.sum() > 0 else float('nan')
print(f"\n  [Model Quality] XGBoost Promo→Loyal Propensity AUC = {auc_p8:.4f}")
 
# Feature importance
fi_p8 = pd.DataFrame({
    'feature':    feature_cols_p8,
    'importance': xgb_p8.feature_importances_
}).sort_values('importance', ascending=False)
print("\n  Feature Importance (tín hiệu dự báo chuyển đổi):")
for _, row in fi_p8.iterrows():
    bar = "█" * int(row['importance'] * 40)
    print(f"    {row['feature']:30s}: {bar} ({row['importance']:.3f})")
 
# Predict trên toàn bộ promo customers
promo_customers_clean = promo_customers.dropna(subset=feature_cols_p8)
promo_customers_clean = promo_customers_clean.copy()
promo_customers_clean['propensity_loyal'] = xgb_p8.predict_proba(
    promo_customers_clean[feature_cols_p8]
)[:, 1]
 
# ------------------------------------------------------------------
# Bước 4: Tính Revenue Uplift nếu chuyển được X% Promo → Loyal
# ------------------------------------------------------------------
ltv_loyal = orders.merge(
    promo_history[promo_history['is_promo_buyer'] == 0][['customer_id']],
    on='customer_id', how='inner'
)['payment_value'].mean()
 
ltv_promo = orders.merge(
    promo_history[promo_history['is_promo_buyer'] == 1][['customer_id']],
    on='customer_id', how='inner'
)['payment_value'].mean()
 
ltv_gap = ltv_loyal - ltv_promo if not (np.isnan(ltv_loyal) or np.isnan(ltv_promo)) else 0
 
# Phân tier propensity
HIGH_PROP  = 0.6
MED_PROP   = 0.3
 
promo_customers_clean['nurture_tier'] = pd.cut(
    promo_customers_clean['propensity_loyal'],
    bins=[-np.inf, MED_PROP, HIGH_PROP, np.inf],
    labels=['Low (< 30%)', 'Medium (30-60%)', 'High (> 60%)']
)
 
tier_summary = promo_customers_clean.groupby('nurture_tier', observed=True).agg(
    n_customers       = ('customer_id', 'count'),
    avg_propensity    = ('propensity_loyal', 'mean'),
    avg_net_revenue   = ('net_revenue_total', 'mean'),
).reset_index()
 
print("\n" + "="*70)
print("PRESCRIPTIVE: PHÂN TIER NURTURE PROMO → LOYAL")
print("="*70)
print(f"\n  LTV gap (Loyal vs Promo): {ltv_gap:,.0f} VNĐ / đơn hàng")
print()
print(f"  {'Tier':<20} {'KH':<10} {'Propensity TB':<18} {'Rev TB':<15} {'Uplift Tiềm năng'}")
print("  " + "-"*75)
 
for _, row in tier_summary.iterrows():
    n   = int(row['n_customers'])
    p   = row['avg_propensity']
    rev = row['avg_net_revenue']
    # Uplift = số KH × propensity × LTV gap (kỳ vọng nếu convert)
    uplift = n * p * ltv_gap
    print(f"  {str(row['nurture_tier']):<20} {n:<10} {p*100:.1f}%{'':10} {rev:>10,.0f}     {uplift:>15,.0f} VNĐ")
 
total_uplift = (promo_customers_clean['propensity_loyal'] * ltv_gap).sum()
print(f"\n  Tổng Revenue Uplift tiềm năng (toàn bộ Promo pool): {total_uplift:,.0f} VNĐ")
 
high_tier = promo_customers_clean[promo_customers_clean['nurture_tier'] == 'High (> 60%)']
print(f"""
  HÀNH ĐỘNG:
  → Nhóm High Propensity ({len(high_tier):,} KH): Trigger nurture sequence
    KHÔNG dùng discount. Thay bằng: Loyalty Points, Early Access, Content.
    (Discount chỉ củng cố thêm hành vi promo-dependent)
  → Nhóm Medium ({tier_summary[tier_summary['nurture_tier']=='Medium (30-60%)']['n_customers'].sum():,} KH):
    Email drip campaign nhẹ, theo dõi thêm 1 chu kỳ mua.
  → Nhóm Low: Không đầu tư thêm.
""")



PILLAR 6: PROMO → LOYAL CONVERSION PIPELINE

  Tỷ lệ Promo → Loyal thực tế trong 90 ngày: 3.86%
  Số KH Promo phân tích: 67,116
  ⚠️  Label imbalance: 3.9% positive. Dùng scale_pos_weight.

  [Model Quality] XGBoost Promo→Loyal Propensity AUC = 0.7440

  Feature Importance (tín hiệu dự báo chuyển đổi):
    total_orders                  : █████████████████████████ (0.632)
    net_revenue_total             : ████ (0.121)
    non_promo_orders              : ██ (0.059)
    avg_discount_rate             : ██ (0.054)
    days_since_last               : █ (0.049)
    categories_bought             : █ (0.047)
    promo_rate                    : █ (0.036)

PRESCRIPTIVE: PHÂN TIER NURTURE PROMO → LOYAL

  LTV gap (Loyal vs Promo): 2,757 VNĐ / đơn hàng

  Tier                 KH         Propensity TB      Rev TB          Uplift Tiềm năng
  ---------------------------------------------------------------------------
  Low (< 30%)          25408      15.7%               49,669          11,009,173 

# PILLAR 7: GROSS PROFIT MIX — NÂNG TỶ LỆ KH BALANCED BUYER

In [9]:
# =============================================================================
# PILLAR 9: GROSS PROFIT MIX — NÂNG TỶ LỆ KH BALANCED BUYER
# =============================================================================
# Insight Q4: Nhóm Balanced (mua nhiều category) có AOV cao nhất, vượt
#             Everyday về doanh thu (Kruskal-Wallis confirmed).
# Mục tiêu: Trong số KH Everyday, ai có propensity cao để mua thêm
#           category mới? → Target bundling offer cho đúng người.
#
# Định nghĩa:
#   Everyday buyer  : Chỉ mua 1 category duy nhất trong lịch sử
#   Balanced buyer  : Đã mua ≥ 2 category khác nhau
#   Label           : KH Everyday → trong 90 ngày tiếp theo có mua
#                     thêm 1 category mới không?
# =============================================================================
print("\n\n" + "="*80)
print("PILLAR 7: GROSS PROFIT MIX — EVERYDAY → BALANCED BUYER")
print("="*80)
 
CUTOFF_P9 = MAX_DATE - pd.Timedelta(days=90)
 
# ------------------------------------------------------------------
# Bước 1: Tính profile KH tại feature window (trước cutoff)
# ------------------------------------------------------------------
oi_all = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date']], on='order_id', how='left'
)
oi_all['order_date'] = pd.to_datetime(oi_all['order_date'])
 
oi_feat = oi_all[oi_all['order_date'] < CUTOFF_P9].copy()
oi_lbl  = oi_all[
    (oi_all['order_date'] >= CUTOFF_P9) &
    (oi_all['order_date'] <= MAX_DATE)
].copy()
 
cust_profile = oi_feat.groupby('customer_id').agg(
    n_orders              = ('order_id', 'nunique'),
    n_categories          = ('category', 'nunique'),
    categories_list       = ('category', lambda x: set(x)),
    total_revenue         = ('net_revenue', 'sum'),
    avg_order_value       = ('net_revenue', 'mean'),
    avg_discount_rate     = ('discount_amount', lambda x:
                             x.sum() / (oi_feat.loc[x.index, 'unit_price'] *
                                        oi_feat.loc[x.index, 'quantity']).sum()
                             if (oi_feat.loc[x.index, 'unit_price'] *
                                 oi_feat.loc[x.index, 'quantity']).sum() > 0 else 0),
    days_as_customer      = ('order_date', lambda x:
                             (x.max() - x.min()).days + 1),
    last_order_date       = ('order_date', 'max'),
).reset_index()
 
cust_profile['days_since_last'] = (
    CUTOFF_P9 - pd.to_datetime(cust_profile['last_order_date'])
).dt.days
 
# Phân loại buyer type tại cutoff
cust_profile['buyer_type'] = np.where(
    cust_profile['n_categories'] == 1, 'Everyday', 'Balanced'
)
 
# Chỉ xét KH Everyday để dự báo
everyday_customers = cust_profile[cust_profile['buyer_type'] == 'Everyday'].copy()
 
# ------------------------------------------------------------------
# Bước 2: Gán nhãn — KH Everyday có mua thêm category mới trong 90 ngày?
# ------------------------------------------------------------------
if not oi_lbl.empty:
    # Category đã mua trong label window
    new_cat_in_label = oi_lbl.groupby('customer_id')['category'].apply(set).reset_index()
    new_cat_in_label.columns = ['customer_id', 'new_categories']
 
    everyday_customers = everyday_customers.merge(
        new_cat_in_label, on='customer_id', how='left'
    )
    everyday_customers['new_categories'] = everyday_customers['new_categories'].apply(
        lambda x: x if isinstance(x, set) else set()
    )
    # Label = 1 nếu mua ít nhất 1 category KHÁC với category hiện tại
    everyday_customers['expanded_to_balanced'] = everyday_customers.apply(
        lambda r: int(len(r['new_categories'] - r['categories_list']) > 0), axis=1
    )
else:
    everyday_customers['expanded_to_balanced'] = 0
    print("  ⚠️  Không có data trong label window.")
 
expand_rate = everyday_customers['expanded_to_balanced'].mean()
print(f"\n  Tỷ lệ Everyday tự nhiên mua thêm category mới (90 ngày): {expand_rate*100:.2f}%")
print(f"  Số KH Everyday phân tích: {len(everyday_customers):,}")
 
# ------------------------------------------------------------------
# Bước 3: Predictive — XGBoost propensity mua thêm category
# ------------------------------------------------------------------
feature_cols_p9 = [
    'n_orders', 'total_revenue', 'avg_order_value',
    'avg_discount_rate', 'days_as_customer', 'days_since_last'
]
 
p9_df = everyday_customers[feature_cols_p9 + ['expanded_to_balanced']].dropna()
X_p9  = p9_df[feature_cols_p9]
y_p9  = p9_df['expanded_to_balanced']
 
pos_rate_p9 = y_p9.mean()
spw_p9 = (1 - pos_rate_p9) / pos_rate_p9 if 0 < pos_rate_p9 < 1 else 1.0
 
X_tr9, X_te9, y_tr9, y_te9 = train_test_split(
    X_p9, y_p9, test_size=0.2, random_state=42,
    stratify=y_p9 if y_p9.sum() > 5 else None
)
 
xgb_p9 = XGBClassifier(
    n_estimators=150, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw_p9, eval_metric='auc', random_state=42
)
xgb_p9.fit(X_tr9, y_tr9)
 
y_prob_p9 = xgb_p9.predict_proba(X_te9)[:, 1]
auc_p9    = roc_auc_score(y_te9, y_prob_p9) if y_te9.sum() > 0 else float('nan')
print(f"\n  [Model Quality] XGBoost Everyday→Balanced Propensity AUC = {auc_p9:.4f}")
 
# Feature importance
fi_p9 = pd.DataFrame({
    'feature':    feature_cols_p9,
    'importance': xgb_p9.feature_importances_
}).sort_values('importance', ascending=False)
print("\n  Feature Importance:")
for _, row in fi_p9.iterrows():
    bar = "█" * int(row['importance'] * 40)
    print(f"    {row['feature']:30s}: {bar} ({row['importance']:.3f})")
 
# Predict propensity trên toàn bộ Everyday customers
everyday_clean = everyday_customers.dropna(subset=feature_cols_p9).copy()
everyday_clean['propensity_balanced'] = xgb_p9.predict_proba(
    everyday_clean[feature_cols_p9]
)[:, 1]
 
# ------------------------------------------------------------------
# Bước 4: Tính Gross Profit Incremental nếu nâng tỷ lệ Balanced
# ------------------------------------------------------------------
aov_balanced = cust_profile[cust_profile['buyer_type'] == 'Balanced']['avg_order_value'].mean()
aov_everyday = cust_profile[cust_profile['buyer_type'] == 'Everyday']['avg_order_value'].mean()
aov_lift_per_convert = aov_balanced - aov_everyday
 
# Số lần mua trung bình / năm (proxy từ lịch sử)
avg_orders_per_year = cust_profile['n_orders'].mean() / \
    max((MAX_DATE - MIN_DATE).days / 365, 1)
 
# GP uplift = (số KH convert) × avg_orders_year × AOV lift
# (Giả định gross margin ~40% là mức phổ biến ngành thời trang)
GROSS_MARGIN = 0.40
 
# Phân tier propensity
everyday_clean['propensity_tier'] = pd.cut(
    everyday_clean['propensity_balanced'],
    bins=[-np.inf, 0.3, 0.6, np.inf],
    labels=['Low', 'Medium', 'High']
)
 
tier_p9 = everyday_clean.groupby('propensity_tier', observed=True).agg(
    n_customers       = ('customer_id', 'count'),
    avg_propensity    = ('propensity_balanced', 'mean'),
    avg_revenue       = ('total_revenue', 'mean'),
).reset_index()
 
print("\n" + "="*70)
print("PRESCRIPTIVE: PHÂN TIER BUNDLE TARGETING")
print("="*70)
print(f"\n  AOV Balanced buyer : {aov_balanced:,.0f} VNĐ")
print(f"  AOV Everyday buyer : {aov_everyday:,.0f} VNĐ")
print(f"  AOV lift nếu convert: +{aov_lift_per_convert:,.0f} VNĐ / đơn")
print(f"  Avg orders/year    : {avg_orders_per_year:.1f} đơn")
print()
print(f"  {'Tier':<12} {'KH':<10} {'Propensity TB':<18} {'GP Uplift / năm'}")
print("  " + "-"*58)
 
total_gp_uplift = 0
for _, row in tier_p9.iterrows():
    n   = int(row['n_customers'])
    p   = row['avg_propensity']
    # Kỳ vọng: n × p KH convert → mỗi KH tăng avg_orders lần × aov_lift × margin
    gp_uplift = n * p * avg_orders_per_year * aov_lift_per_convert * GROSS_MARGIN
    total_gp_uplift += gp_uplift
    print(f"  {str(row['propensity_tier']):<12} {n:<10} {p*100:.1f}%{'':10} {gp_uplift:>15,.0f} VNĐ")
 
print(f"\n  Tổng GP Uplift tiềm năng / năm: {total_gp_uplift:,.0f} VNĐ")
 
# Xác định category "mồi nhử" tốt nhất — category nào hay được mua cùng Streetwear nhất
if 'Streetwear' in oi_feat['category'].values:
    streetwear_orders = oi_feat[oi_feat['category'] == 'Streetwear']['order_id'].unique()
    cross_cat = oi_feat[
        (oi_feat['order_id'].isin(streetwear_orders)) &
        (oi_feat['category'] != 'Streetwear')
    ]['category'].value_counts()
 
    if not cross_cat.empty:
        top_cross_cat = cross_cat.index[0]
        print(f"\n  Category hay được mua kèm Streetwear nhất: [{top_cross_cat}]")
        print(f"  → Đây là 'mồi nhử' tối ưu cho bundle offer.")
 
high_p9 = everyday_clean[everyday_clean['propensity_tier'] == 'High']
print(f"""
  HÀNH ĐỘNG:
  → {len(high_p9):,} KH Everyday có propensity cao (>60%):
    Trigger Bundle Offer: "Mua Streetwear → Giảm X% {top_cross_cat if not cross_cat.empty else 'Outdoor'}"
    (Tactic đã được validate bằng Lift > 1 từ Q3)
 
  → KPI cứng cho team Sales/MKT:
    Mục tiêu chuyển đổi: {int(len(high_p9) * 0.3):,} KH Everyday → Balanced trong 2 quý tới
    = Tương đương GP Uplift: {int(len(high_p9) * 0.3) * avg_orders_per_year * aov_lift_per_convert * GROSS_MARGIN:,.0f} VNĐ / năm
 
  → Đo lường: Track tỷ lệ "2nd category purchase rate" theo tháng.
    Alert nếu tỷ lệ < {expand_rate*100*1.5:.1f}% (1.5× baseline tự nhiên).
""")



PILLAR 7: GROSS PROFIT MIX — EVERYDAY → BALANCED BUYER

  Tỷ lệ Everyday tự nhiên mua thêm category mới (90 ngày): 0.72%
  Số KH Everyday phân tích: 34,748

  [Model Quality] XGBoost Everyday→Balanced Propensity AUC = 0.5189

  Feature Importance:
    days_as_customer              : █████████ (0.231)
    total_revenue                 : ███████ (0.180)
    days_since_last               : ██████ (0.168)
    avg_discount_rate             : █████ (0.144)
    n_orders                      : █████ (0.142)
    avg_order_value               : █████ (0.135)

PRESCRIPTIVE: PHÂN TIER BUNDLE TARGETING

  AOV Balanced buyer : 21,426 VNĐ
  AOV Everyday buyer : 24,363 VNĐ
  AOV lift nếu convert: +-2,937 VNĐ / đơn
  Avg orders/year    : 0.7 đơn

  Tier         KH         Propensity TB      GP Uplift / năm
  ----------------------------------------------------------
  Low          12219      20.3%                -1,977,255 VNĐ
  Medium       20082      42.4%                -6,777,673 VNĐ
  High      

## Bo sung: Predictive & Prescriptive theo `notebooks/analysis_gaps.md`

Cac o code ben duoi mo rong cac khoang phan tich con thieu (Streetwear, COVID/CR, KM, RFM/churn, dia ly, Premium, marketing). **Chay notebook theo thu tu tu tren xuong** de bien `tx_data`, `order_items`, `orders`, `web_traffic`, `MAX_DATE` va (cho muc churn nang cao) mo hinh `bgf`, `rfm_repeat_reset` tu Pillar 1 duoc khoi tao.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

print('\n' + '=' * 80)
print('GAP 1 — Streetwear dependency & GAP 5 — Dia ly (phan 1)')
print('=' * 80)

# --- 1A. Ty trong Streetwear theo nam + Spearman + ngoai suy 2023–2025 ---
oi_dates = order_items.merge(orders[['order_id', 'order_date']], on='order_id', how='left')
oi_dates['year'] = oi_dates['order_date'].dt.year
yr_cat = oi_dates.groupby(['year', 'category'])['net_revenue'].sum().unstack(fill_value=0)
if 'Streetwear' in yr_cat.columns:
    yr_cat['street_share'] = yr_cat['Streetwear'] / yr_cat.sum(axis=1).replace(0, np.nan)
    ys = yr_cat.loc[2012:2022, 'street_share'].dropna()
    rho, p_sp = spearmanr(ys.index.values, ys.values)
    slope, intercept = np.polyfit(ys.index.values.astype(float), ys.values.astype(float), 1)
    fut_years = np.array([2023, 2024, 2025])
    fut_share = slope * fut_years + intercept
    print('\n[Predictive] Ty trong doanh thu Streetwear theo nam (2012–2022)')
    print(f'  Spearman rho (nam vs ty trong) = {rho:+.4f}  p-value = {p_sp:.4g}')
    print('  Ngoai suy tuyen tinh ty trong (khong can thiep):')
    for y, s in zip(fut_years, fut_share):
        print(f'    {y}: {100 * s:.2f}%')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ys.index, 100 * ys.values, 'o-', label='Thuc te 2012–2022')
    ax.plot(fut_years, 100 * fut_share, 's--', label='Ngoai suy 2023–2025')
    ax.set_xlabel('Nam')
    ax.set_ylabel('Ty trong Streetwear (%)')
    ax.legend()
    ax.set_title('Streetwear share of revenue — trend & linear forecast')
    plt.tight_layout()
    plt.show()
else:
    print('  Khong co cot Streetwear trong du lieu.')

# --- 1B. Cross-sell / bundle: Outdoor + Streetwear trong cung don ---
oi_oid = order_items.merge(orders[['order_id', 'customer_id', 'order_date']], on='order_id', how='left')
by_order = oi_oid.groupby('order_id').agg(
    rev=('net_revenue', 'sum'),
    cats=('category', lambda x: set(x.dropna())),
).reset_index()
by_order['has_sw'] = by_order['cats'].apply(lambda s: 'Streetwear' in s)
by_order['has_out'] = by_order['cats'].apply(lambda s: 'Outdoor' in s)
by_order['both'] = by_order['has_sw'] & by_order['has_out']
sw_only = by_order['has_sw'] & ~by_order['both']
base_aov_sw = by_order.loc[sw_only, 'rev'].mean()
bundle_aov = by_order.loc[by_order['both'], 'rev'].mean()
n_both = by_order['both'].sum()
n_sw = by_order['has_sw'].sum()
print('\n[Predictive/Prescriptive] Basket Outdoor + Streetwear')
print(f'  So don co Streetwear: {n_sw:,} | don co ca Outdoor: {n_both:,} ({100 * n_both / max(n_sw, 1):.2f}%)')
if n_both > 0 and sw_only.sum() > 0:
    lift = bundle_aov / base_aov_sw - 1
    print(f'  AOV don chi Streetwear (xap xi): {base_aov_sw:,.0f} VND')
    print(f'  AOV don co Streetwear + Outdoor: {bundle_aov:,.0f} VND')
    print(f'  Uplift AOV neu bundle thanh cong: {100 * lift:.1f}%')
    if 'Streetwear' in yr_cat.columns:
        ys_last = float(yr_cat.loc[2012:2022, 'street_share'].dropna().iloc[-1])
        x_pct = 5.0
        y_share = ys_last * (1 - x_pct / 100.0)
        print('\n[Prescriptive] Vi du dinh luong (gia dinh don gian):')
        print(f'  Neu cross-sell lam AOV tang them {x_pct:.0f}% va ty trong Streetwear giam tuong duong ~{x_pct:.0f}% tuyet doi tren co so nam gan nhat,')
        print(f'  ty trong Streetwear (nam 2022) xap xi con ~{100 * y_share:.1f}% cua muc 2022 (can mo hinh don cat de khoa chinh xac hon).')

# --- 5A. Tang truong KH theo region + thi phan 2025 ---
cust_geo = customers.merge(geography[['zip', 'region']], on='zip', how='left')
first_order = orders.sort_values('order_date').groupby('customer_id')['order_date'].min().reset_index()
first_order = first_order.merge(cust_geo[['customer_id', 'region']], on='customer_id', how='left')
first_order['year'] = first_order['order_date'].dt.year
new_by = first_order.groupby(['year', 'region']).size().unstack(fill_value=0)
regions = [c for c in ['East', 'Central', 'West'] if c in new_by.columns]
if regions:
    new_by = new_by[regions]
    last_fit = new_by.loc[2018:2022]
    pred_2025 = {}
    for r in regions:
        yv = last_fit[r].values
        xv = last_fit.index.values.astype(float)
        m, b = np.polyfit(xv, yv, 1)
        pred_2025[r] = max(0.0, m * 2025 + b)
    tot_2025 = sum(pred_2025.values())
    print('\n[Predictive] Ngoai suy so KH moi (first order) nam 2025 theo khu vuc (tuyen tinh 2018–2022)')
    for r in regions:
        print(f'  {r}: du bao ~{pred_2025[r]:,.0f} KH moi | thi phan ~{100 * pred_2025[r] / tot_2025:.1f}%')
    rev_cust = oi_oid.merge(cust_geo[['customer_id', 'region']], on='customer_id', how='left')
    rev_by = rev_cust.groupby('region').agg(rev=('net_revenue', 'sum'), n_cust=('customer_id', 'nunique'))
    east_pen = (rev_by.loc['East', 'rev'] / rev_by.loc['East', 'n_cust']) if 'East' in rev_by.index else np.nan
    if 'West' in rev_by.index and pd.notna(east_pen):
        west_c = rev_by.loc['West', 'n_cust']
        west_rev = rev_by.loc['West', 'rev']
        potential = east_pen * west_c - west_rev
        print('\n[Predictive] Revenue tiem nang West neu penetration (rev/KH) bang East')
        print(f'  Hien tai West: {west_rev:,.0f} VND / {west_c:,.0f} KH')
        print(f'  Ky vong neu bang East: {east_pen * west_c:,.0f} VND | chenh ~{potential:,.0f} VND (tich luy lich su, thuan tuyen)')
    x_re = 10.0
    if 'East' in rev_by.index and 'West' in rev_by.index:
        east_rev = rev_by.loc['East', 'rev']
        west_rpc = rev_by.loc['West', 'rev'] / max(rev_by.loc['West', 'n_cust'], 1)
        shift_rev = (x_re / 100.0) * east_rev * (west_rpc / max(east_rev / max(rev_by.loc['East', 'n_cust'], 1), 1))
        print('\n[Prescriptive] Vi du tai phan bo marketing: chuyen ~{:.0f}% "hieu qua" tu East sang West'.format(x_re))
        print(f'  (Baseline don gian) revenue ky vong tang them ~{shift_rev:,.0f} VND neu West hien tai co revenue-per-customer = baseline West.')
else:
    print('  Thieu cot region hoac khong du East/Central/West.')

In [ ]:
print('\n' + '=' * 80)
print('GAP 2 — COVID / Conversion rate & GAP 3 — Khuyen mai')
print('=' * 80)

if 'oi_dates' not in dir():
    oi_dates = order_items.merge(orders[['order_id', 'order_date']], on='order_id', how='left')
web_traffic['year'] = web_traffic['date'].dt.year
web_traffic['month'] = web_traffic['date'].dt.month
orders_per_day = orders.groupby(orders['order_date'].dt.date)['order_id'].count().reset_index()
orders_per_day.columns = ['date', 'orders_count']
orders_per_day['date'] = pd.to_datetime(orders_per_day['date'])
so = web_traffic.merge(orders_per_day, on='date', how='left').fillna({'orders_count': 0})
tot_s = so.groupby('date')['sessions'].transform('sum')
so['cr_day'] = (so['orders_count'] * (so['sessions'] / tot_s.replace(0, np.nan))).fillna(0) / so['sessions'].replace(0, np.nan)
so['cr_day'] = so['cr_day'].fillna(0).clip(0, 1)
cr_year = so.groupby('year')['cr_day'].mean()
post = cr_year[cr_year.index >= 2019]
if len(post) >= 2:
    y2022 = float(cr_year.loc[2022]) if 2022 in cr_year.index else float(post.iloc[-1])
    y2021 = float(cr_year.loc[2021]) if 2021 in cr_year.index else float(post.iloc[-2])
    delta_cr_per_year = y2022 - y2021
    print('\n[Predictive] CR trung binh theo nam (sau 2019):')
    print(post.to_string())
    print(f'  Thay doi CR 2021->2022: {100 * delta_cr_per_year:+.4f} diem phan tram (pp) / nam (tai lieu tham chieu +3.6%%/nam la ty le tuong doi)')
    target_cr = 0.00926
    cr_now = float(cr_year.loc[2022]) if 2022 in cr_year.index else float(post.iloc[-1])
    if abs(delta_cr_per_year) > 1e-8:
        years_need = (target_cr - cr_now) / delta_cr_per_year
        print(f'  Ngoai suy tuyen tinh theo toc do 2021-2022: can ~{years_need:.1f} nam de dat CR={100*target_cr:.3f}% (gia dinh xu huong tuyen tinh tiep tuc, can than voi ngoai suy).')
    else:
        print('  Khong du bien dong nam de ngoai suy.')

# --- 2B. Seasonality CR theo thang (toan bo lich su) ---
cr_month = so.groupby('month')['cr_day'].mean().reindex(range(1, 13))
best_m = int(cr_month.idxmax())
worst_m = int(cr_month.idxmin())
print('\n[Predictive] CR theo thang (trung binh toan bo quan sat):')
print(f'  Thang phuc hoi manh nhat (CR TB cao): {best_m} ({100 * cr_month.max():.4f}%)')
print(f'  Thang yeu nhat: {worst_m} ({100 * cr_month.min():.4f}%)')
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(cr_month.index, 100 * cr_month.values)
ax.set_xlabel('Thang')
ax.set_ylabel('CR trung binh (%)')
ax.set_title('Seasonality CR theo thang')
plt.tight_layout()
plt.show()

# --- 2C. Doanh thu them neu CR 0.351% -> 0.5% (giu sessions & AOV) ---
s_post = so[so['year'] >= 2019]['sessions'].sum()
aov_post = orders[orders['order_date'].dt.year >= 2019]['payment_value'].mean()
cr_base = 0.00351
cr_new = 0.00500
rev_base = s_post * cr_base * aov_post
rev_new = s_post * cr_new * aov_post
print('\n[Prescriptive] Tac dong doanh thu neu CR tang (giu sessions & AOV post-2019)')
print(f'  Sessions post-2019: {s_post:,.0f} | AOV TB: {aov_post:,.0f} VND')
print(f'  Doanh thu ky vong tai CR={100*cr_base:.3f}%: {rev_base:,.0f} VND')
print(f'  Doanh thu ky vong tai CR={100*cr_new:.2f}%: {rev_new:,.0f} VND')
print(f'  Chenh lech: +{rev_new - rev_base:,.0f} VND')
print('  De xuat: A/B test checkout (guest vs login), giam buoc thanh toan, thu nghiem one-page checkout, toi uu mobile.')

# --- 3A. Seasonality doanh thu theo thang (2012–2022) ---
oi_dates['month'] = oi_dates['order_date'].dt.month
mrev = oi_dates.groupby('month')['net_revenue'].sum()
ranked = mrev.sort_values(ascending=False)
print('\n[Predictive] Doanh thu theo thang (tong 2012–2022) — top / bottom')
print('  Top 3 thang:', ', '.join(f'{int(m)} ({mrev[m]:,.0f} VND)' for m in ranked.head(3).index))
print('  Bottom 3 thang:', ', '.join(f'{int(m)} ({mrev[m]:,.0f} VND)' for m in ranked.tail(3).sort_values().index))
print('  Goi y: uu tien KM o thang manh season neu muc tieu la doanh thu; tranh dot bien o thang yeu neu chi phi co dinh.')

# --- 3B. LTV / doanh thu lich su: nhom "san KM" vs it KM ---
oi_c = oi_oid.copy()
oi_c['has_promo_line'] = oi_c['promo_id'].notna() | oi_c['promo_id_2'].notna()
pr = oi_c.groupby('customer_id').agg(
    promo_rate=('has_promo_line', 'mean'),
    spend=('net_revenue', 'sum'),
    n_ord=('order_id', 'nunique'),
).reset_index()
hi = pr['promo_rate'] >= 0.5
lo = pr['promo_rate'] <= 0.1
print('\n[Predictive] So sanh nhom san KM (>=50% dong co promo) vs nhom it promo (<=10%)')
print(f'  TB tong spend / KH — san KM: {pr.loc[hi, 'spend'].mean():,.0f} | it promo: {pr.loc[lo, 'spend'].mean():,.0f} VND')
print(f'  TB so don — san KM: {pr.loc[hi, 'n_ord'].mean():.2f} | it promo: {pr.loc[lo, 'n_ord'].mean():.2f}')

# --- 3C. Giam gia thang 12 vs thang tot nhat + mo phong retention ---
oi_c['ym'] = oi_c['order_date'].dt.to_period('M')
disc_m = oi_c.groupby(oi_c['order_date'].dt.month)['discount_amount'].sum()
rev_m = oi_c.groupby(oi_c['order_date'].dt.month)['net_revenue'].sum() + disc_m
dec_disc = disc_m.get(12, 0)
avg_disc_other = disc_m.drop(index=12, errors='ignore').mean()
print('\n[Prescriptive] Giam gia theo thang (tong toan ky)')
print(f'  Tong giam gia thang 12: {dec_disc:,.0f} VND | TB cac thang khac: {avg_disc_other:,.0f} VND / thang')
waste_proxy = max(0.0, dec_disc - avg_disc_other)
print(f'  Uoc tinh "lang phi" tuong doi (thang 12 - TB thang khac): {waste_proxy:,.0f} VND (dinh nghia tho, can gan voi ngan sach KM that)')
ret_lift = 0.15
ltv_proxy = pr.loc[lo, 'spend'].mean() - pr.loc[hi, 'spend'].mean()
sim_ltv = waste_proxy * ret_lift
print(f'  Mo phong: neu chuyen {ret_lift*100:.0f}% phan chenh len tren sang retention, va moi VND doi duoc {ret_lift:.2f}x "LTV-style" don gian -> ~{sim_ltv:,.0f} VND gia tri tich luy tuyet doi (chi mang tinh minh hoa).')

In [ ]:
print('\n' + '=' * 80)
print('GAP 4 — Churn / RFM | GAP 6 — Premium | GAP 7 — Marketing')
print('=' * 80)

# --- 4A. Hibernating + BG/NBD: ky vong "churn hoan toan" 12 thang ---
last_buy = tx_data.groupby('customer_id')['order_date'].max()
days_since_last = (MAX_DATE - last_buy).dt.days
hibernating_ids = days_since_last[days_since_last > 180].index
if 'bgf' in dir() and 'rfm_repeat_reset' in dir():
    hib = rfm_repeat_reset[rfm_repeat_reset['customer_id'].isin(hibernating_ids)].copy()
    if len(hib) > 0:
        pa = bgf.probability_alive(hib['frequency'], hib['recency'], hib['T'])
        prob_dead = 1.0 - pa
        print('\n[Predictive] Nhom Hibernating (recency > 180 ngay tinh tu MAX_DATE, repeat buyers trong Pillar 1)')
        print(f'  So KH (repeat, trong rfm_repeat_reset): {len(hib):,}')
        print(f'  Tong (1 - P alive)) — ky vong so "mat hoan toan" tham chieu BG/NBD: {prob_dead.sum():,.0f}')
        print(f'  So KH co P(alive) < 0.5: {(pa < 0.5).sum():,}')
    else:
        print('\n[Predictive] Khong co repeat buyer nao thoa Hibernating trong bang da fit.')
else:
    print('\n[Predictive] Chua co bgf/rfm_repeat_reset — chay Pillar 1 truoc de xem muc Hibernating.')

# --- 4B. RFM + CLV du bao theo phan khuc (Champions / Loyal / At Risk / Promising) ---
snap = MAX_DATE
rf = tx_data.groupby('customer_id').agg(
    last=('order_date', 'max'),
    F=('order_id', 'nunique'),
    M=('payment_value', 'sum'),
).reset_index()
rf['R_days'] = (snap - rf['last']).dt.days

def q5_labels(s, low_is_bad=True):
    s = s.astype(float)
    lab = pd.qcut(s.rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    return 6 - lab if low_is_bad else lab

rf['r_s'] = q5_labels(rf['R_days'], low_is_bad=True)
rf['f_s'] = q5_labels(rf['F'].clip(lower=1), low_is_bad=False)
rf['m_s'] = q5_labels(rf['M'], low_is_bad=False)

def seg_row(r):
    if r['r_s'] >= 4 and r['f_s'] >= 4 and r['m_s'] >= 4:
        return 'Champions'
    if r['f_s'] >= 4 and r['m_s'] >= 3:
        return 'Loyal'
    if r['r_s'] <= 2 and r['f_s'] >= 3:
        return 'At Risk'
    if r['r_s'] >= 4 and r['f_s'] <= 2:
        return 'Promising'
    return 'Other'

rf['rfm_segment'] = rf.apply(seg_row, axis=1)
if 'all_customers' in dir() and 'rfm_repeat_reset' in dir():
    ltv_map = rfm_repeat_reset.set_index('customer_id')['predicted_LTV_12m']
    rf['pred_ltv_12m'] = rf['customer_id'].map(ltv_map).fillna(0.0)
else:
    rf['pred_ltv_12m'] = rf['M'] / max((MAX_DATE - MIN_DATE).days / 365.25, 1) * 0.25

idx_seg = ['Champions', 'Loyal', 'At Risk', 'Promising']
seg_sum = rf.groupby('rfm_segment').agg(n=('customer_id', 'count'), clv=('pred_ltv_12m', 'sum')).reindex(idx_seg)
print('\n[Predictive] CLV du bao 12T (BG/GG neu co) theo phan khuc RFM:')
print(seg_sum.to_string())

ar = rf[rf['rfm_segment'] == 'At Risk']
seg_promising = rf[rf['rfm_segment'] == 'Promising']
print('\n[Prescriptive] Uu tien At Risk vs Promising (theo tong CLV du bao 12T)')
if len(ar) and len(seg_promising):
    print(f'  At Risk: {len(ar):,} KH | tong CLV du bao: {ar["pred_ltv_12m"].sum():,.0f} VND')
    print(f'  Promising: {len(seg_promising):,} KH | tong CLV du bao: {seg_promising["pred_ltv_12m"].sum():,.0f} VND')
    if ar['pred_ltv_12m'].sum() >= seg_promising['pred_ltv_12m'].sum():
        print('  Ket luan: uu tien At Risk truoc (tong CLV cao hon trong dinh nghia hien tai).')
    else:
        print('  Ket luan: uu tien Promising truoc (tong CLV cao hon).')
print('\n[Prescriptive] Nguong recency de win-back campaign: > 180 ngay khong mua (da dung cho Hibernating).')

# --- 6. Premium: bien trend, Spearman gia vs tan suat, gross profit 5.9% -> 8% ---
if 'oi_dates' not in dir():
    oi_dates = order_items.merge(orders[['order_id', 'order_date']], on='order_id', how='left')
oi_seg = oi_dates.merge(products[['product_id', 'segment', 'cogs']], on='product_id', how='left')
prem = oi_seg[oi_seg['segment'].astype(str).str.contains('Premium', case=False, na=False)].copy()
if len(prem) > 0:
    prem['gp'] = prem['net_revenue'] - prem['quantity'] * prem['cogs'].fillna(0)
    prem['year'] = prem['order_date'].dt.year
    gpy = prem.groupby('year').agg(rev=('net_revenue', 'sum'), gp=('gp', 'sum'))
    gpy['margin'] = gpy['gp'] / gpy['rev'].replace(0, np.nan)
    print('\n[Predictive] Bien loi nhuan Premium (gross profit / revenue) theo nam:')
    print(gpy['margin'].to_string())
    if len(gpy) >= 2:
        xs = gpy.index.values.astype(float)
        ys = gpy['margin'].values.astype(float)
        m_g, b_g = np.polyfit(xs, ys, 1)
        print(f'  Ngoai suy tuyen tinh bien nam 2023: ~{100 * (m_g * 2023 + b_g):.2f}% (gia dinh xu huong tiep tuc)')
    cust_p = prem.groupby('customer_id').agg(n_ord=('order_id', 'nunique'), avg_price=('unit_price', 'mean'))
    rho2, p2 = spearmanr(cust_p['avg_price'], cust_p['n_ord'])
    print(f'\n[Predictive] Spearman gia TB dong vs so don (Premium buyers): rho = {rho2:+.4f}, p = {p2:.4g}')
    prem_buyers = rf[rf['customer_id'].isin(cust_p.index)]
    tgt = prem_buyers['rfm_segment'].value_counts().head(2)
    print('\n[Prescriptive] Nhom RFM xuat hien nhieu nhat trong Premium buyers:')
    print(tgt.to_string())
    rev_p = prem['net_revenue'].sum()
    gp_p = prem['gp'].sum()
    m0 = gp_p / max(rev_p, 1)
    m1 = 0.08
    print(f'\n[Prescriptive] Tong revenue Premium (lich su): {rev_p:,.0f} VND | bien hien tai ~{100*m0:.2f}%')
    print(f'  Neu bien tang tu {100*m0:.2f}% len 8%: chenh gross profit ~{(m1 - m0) * rev_p:,.0f} VND (tich luy tren lich su, khong phai nam don).')
else:
    print('\n[Predictive] Khong co dong Premium trong order_items.')

# --- 7. Marketing: Paid Search trend, diminishing returns, tai phan bo ---
wt = web_traffic.copy()
wt['year'] = wt['date'].dt.year
paid_y = wt[wt['traffic_source'] == 'paid_search'].groupby('year')['sessions'].sum()
print('\n[Predictive] Paid Search: tong sessions theo nam')
print(paid_y.to_string())
if len(paid_y) >= 3:
    px = paid_y.index.values.astype(float)
    py = paid_y.values.astype(float)
    m_p, b_p = np.polyfit(px, py, 1)
    print(f'  Xu huong sessions/nam (tuyen tinh): tang ~{m_p:,.0f} sessions / nam (neu duong)')
    print('  Neu CPC tang cung ty le voi cau (gia dinh), ROI bien giam khi sessions tang ma orders khong ty le.')

_orders_pd = orders.groupby(orders['order_date'].dt.date)['order_id'].count().reset_index()
_orders_pd.columns = ['date', 'orders_count']
_orders_pd['date'] = pd.to_datetime(_orders_pd['date'])
_aov_post = orders[orders['order_date'].dt.year >= 2019]['payment_value'].mean()

paid_daily = wt[wt['traffic_source'] == 'paid_search'].sort_values('date').copy()
od = _orders_pd.rename(columns={'orders_count': 'o_all'})
paid_daily = paid_daily.merge(od, on='date', how='left').fillna({'o_all': 0})
tot_s_day = wt.groupby('date')['sessions'].transform('sum')
paid_daily['share'] = paid_daily['sessions'] / tot_s_day.replace(0, np.nan)
paid_daily['o_paid'] = paid_daily['o_all'] * paid_daily['share']
paid_daily = paid_daily.sort_values('date')
paid_daily['cs'] = paid_daily['sessions'].cumsum()
paid_daily['co'] = paid_daily['o_paid'].cumsum()
paid_daily['marg_o'] = paid_daily['co'].diff() / paid_daily['sessions'].replace(0, np.nan)
thr = paid_daily[(paid_daily['cs'] > paid_daily['sessions'].sum() * 0.5) & paid_daily['marg_o'].notna()]
if len(thr):
    plateau = thr.loc[thr['marg_o'].idxmin(), 'cs']
    print(f'\n[Predictive] Diminishing returns (cumulative paid sessions): marginal orders/session thap tai cum_sessions ~{plateau:,.0f}')

soc_y = wt[wt['traffic_source'] == 'social_media'].groupby('year')['sessions'].sum()
org_y = wt[wt['traffic_source'] == 'organic_search'].groupby('year')['sessions'].sum()
print('\n[Prescriptive] Vi du tai phan bo: giam Paid Search 15% sessions, chuyen 10% sang Social, 5% Organic (proxy)')
to_soc = 0.10
to_org = 0.05
last_p = float(paid_y.iloc[-1])
_so_cr = wt.merge(_orders_pd, on='date', how='left').fillna({'orders_count': 0})
_tot_d = _so_cr.groupby('date')['sessions'].transform('sum')
_so_cr['cr_row'] = (_so_cr['orders_count'] * (_so_cr['sessions'] / _tot_d.replace(0, np.nan))).fillna(0) / _so_cr['sessions'].replace(0, np.nan)
cr_paid_mean = _so_cr[_so_cr['traffic_source'] == 'paid_search']['cr_row'].mean()
rev_proxy = float(cr_paid_mean * _aov_post * last_p)
uplift_guess = rev_proxy * 0.05 * (to_soc + to_org)
print(f'  Paid sessions nam gan nhat: {last_p:,.0f} | revenue proxy kenh paid (tho): ~{rev_proxy:,.0f} VND')
print(f'  Uplift ky vong rat don gian (+5%% hieu qua tren phan session chuyen): ~{uplift_guess:,.0f} VND (minh hoa).')